# 02 — Agente basado en modelos (Aspiradora con memoria)

**Idea clave:** un agente *basado en modelos* mantiene un **estado interno**:
recuerda lo que ya vio o hizo (mapa, historial, etc.).

## Caso vida diaria
Una **aspiradora robot**:
- no ve todo el cuarto al mismo tiempo,
- necesita “memoria” de dónde hay obstáculos y qué ya limpió.

Simulamos un cuarto con una cuadrícula:
- `1` = obstáculo (mueble/pared)
- `0` = libre
- el robot construye su “creencia” del mundo con lo que percibe localmente.


In [ ]:
# ============================================================
# AGENTE BASADO EN MODELOS: ASPIRADORA CON MEMORIA
# ============================================================

import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

# 1) Creamos un mapa del cuarto (0 = libre, 1 = obstáculo)
mapa_real = np.zeros((10,10), dtype=int)
mapa_real[2, 2:6] = 1          # "mueble" horizontal
mapa_real[6:9, 7] = 1          # "mueble" vertical
mapa_real[0, :] = 1            # pared arriba
mapa_real[-1, :] = 1           # pared abajo
mapa_real[:, 0] = 1            # pared izquierda
mapa_real[:, -1] = 1           # pared derecha

# 2) Estado interno del agente: lo que "cree" del mundo (inicialmente desconocido)
#    -1 = desconocido, 0 = libre, 1 = obstáculo
creencia = -1 * np.ones_like(mapa_real)

# 3) Posición inicial del robot (fila, columna)
pos = [1, 1]

# 4) Celdas ya "limpiadas"
limpiado = np.zeros_like(mapa_real)

# 5) Función de percepción: ve su celda y vecinos (radio 1)
def percibir(mapa, pos):
    r, c = pos
    r0, r1 = max(0, r-1), min(mapa.shape[0], r+2)
    c0, c1 = max(0, c-1), min(mapa.shape[1], c+2)
    return (r0, r1, c0, c1, mapa[r0:r1, c0:c1])

# 6) Movimientos posibles (arriba/abajo/izq/der)
movs = [(-1,0),(1,0),(0,-1),(0,1)]

def elegir_movimiento(creencia, limpiado, pos):
    r, c = pos

    # a) vecinos desconocidos -> explorar
    for dr, dc in movs:
        nr, nc = r+dr, c+dc
        if creencia[nr, nc] == -1:
            return [nr, nc]

    # b) vecinos libres y no limpiados -> completar limpieza
    for dr, dc in movs:
        nr, nc = r+dr, c+dc
        if creencia[nr, nc] == 0 and limpiado[nr, nc] == 0:
            return [nr, nc]

    # c) cualquier libre -> moverse
    for dr, dc in movs:
        nr, nc = r+dr, c+dc
        if creencia[nr, nc] == 0:
            return [nr, nc]

    # d) si no hay a dónde ir, quedarse
    return [r, c]

# 7) Simulación por pasos
pasos = 60
trayectoria = [tuple(pos)]

for _ in range(pasos):
    # ---- PERCEPCIÓN ----
    r0, r1, c0, c1, vista = percibir(mapa_real, pos)

    # ---- ACTUALIZAR MODELO INTERNO ----
    creencia[r0:r1, c0:c1] = vista

    # ---- ACCIÓN: limpiar si es libre ----
    if mapa_real[pos[0], pos[1]] == 0:
        limpiado[pos[0], pos[1]] = 1

    # ---- DECISIÓN: usar memoria para moverse ----
    nueva_pos = elegir_movimiento(creencia, limpiado, pos)

    # Evitamos “chocar” con obstáculos
    if mapa_real[nueva_pos[0], nueva_pos[1]] == 0:
        pos = nueva_pos

    trayectoria.append(tuple(pos))

# 8) Visualización
plt.figure(figsize=(15,4))

plt.subplot(1,3,1)
plt.imshow(mapa_real)
plt.title("Mapa real (0 libre, 1 obstáculo)")
plt.axis("off")

plt.subplot(1,3,2)
plt.imshow(creencia)
plt.title("Creencia del robot (-1 desconocido)")
plt.axis("off")

plt.subplot(1,3,3)
plt.imshow(limpiado)
plt.title("Zonas limpiadas (1 = limpio)")
plt.axis("off")

plt.show()

print("Primeros 15 pasos de la trayectoria:")
print(trayectoria[:15])


## ¿Cómo se aplica en industria?
- Robots móviles (AGV/AMR) en almacenes.
- Drones que mapean zonas.
- Sistemas que “recuerdan” inventario, fallas, mantenimiento. 
